In [3]:
#import environment variables
from dotenv import load_dotenv
load_dotenv()

True

模型的每次调用是无状态的，不会记得之前说了什么，想要拥有记忆需要将历史信息发给他，这就需要用到prompt template，对某个位置留空，在调用的时候补全

举例如下，在调用时填充 new_message 信息

In [1]:
from langchain.prompts import ChatPromptTemplate


prompt_template=ChatPromptTemplate.from_messages(
    [
        ('system',"you are a chatbot"),
        ('human',"{new_message}")
    ]
)

prompt_template.invoke(
    {'new_message':'How are you?'}
).messages

[SystemMessage(content='you are a chatbot'),
 HumanMessage(content='How are you?')]


传入历史消息也需要留空一个位置，需要使用到 MessagesPlaceholder，从输入的prompt中提取variable_name相同的一个列表，其中包含角色和对话，接下来的调用中prompt就能包含历史对话信息了

In [2]:
from langchain.prompts import (MessagesPlaceholder,
                               ChatPromptTemplate)


prompt_template=ChatPromptTemplate.from_messages(
    [
        ('system',"you are a chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ('human',"{new_message}")
    ]
)

prompt_template.invoke(
    {'chat_history':[
        ('human','Hello'),
        ('ai','Hi')
    ],
    'new_message':'How are you?'}
).messages

[SystemMessage(content='you are a chatbot'),
 HumanMessage(content='Hello'),
 AIMessage(content='Hi'),
 HumanMessage(content='How are you?')]

一种比较简单的思路就是，用一个列表存储所有role:content信息，invoke输入prompt时嵌入进去，在实际应用中最好使用langchain的component来实现，这样对于开发的一致性和拓展性都更好
ChatMessageHistory是一个专门用来存储历史消息的类，能很方便的添加user和ai信息
RunnableWithMessageHistory能用来定义一个包含历史信息的runable，调用时需要传入自己的id，这个id会传给刚刚定义的lambda表达式，来获取一个chat message history

In [13]:
from langchain_openai import ChatOpenAI
from langchain.memory import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

model=ChatOpenAI(
    model='glm-4-plus',
    openai_api_base="https://open.bigmodel.cn/api/paas/v4/",
    max_tokens=300,
    temperature=0.7)

chain=prompt_template|model
chat_history=ChatMessageHistory()

chat_history.add_user_message("你好，我是Chaos，今年22岁，我正在学习llm-agent工程")
chat_history.add_ai_message("你好，我是AI")

chat_history.messages

chain_with_memory=RunnableWithMessageHistory(chain,
                                             lambda x:chat_history,
                                             input_messages_key='new_message',
                                             history_messages_key='chat_history')

print(chain_with_memory.invoke(
    {
        'new_message':'请说出你所知道的关于我的信息',
    },
    config={
        "configurable":{
            "session_id":"unused"
        }
    }
).content)

根据你刚才的介绍，我知道以下关于你的信息：

1. 你的名字是Chaos。
2. 你今年22岁。
3. 你正在学习llm-agent工程。

除此之外，我没有其他关于你的信息。如果你愿意分享更多，我很乐意了解更多关于你的情况！


随着对话次数变多，历史消息的增大导致token数随之增大，可能超过模型的限制，所以总结记忆或者修剪记忆就很有必要

在传给chain之前进行修剪或总结，这意味着需要在chain之前添加一个东西，

In [14]:
def trim_messages(chain_input):
    stored_messages=chat_history.messages
    if len(stored_messages)>2:
        chat_history.clear()
        for message in stored_messages[-2:]:
            chat_history.add_message(message)
    return chain_input

chain_with_trimming=trim_messages|chain_with_memory
print(chain_with_trimming.invoke(
    {
        'new_message':'请说出你所知道的关于我的信息',
    },
    config={
        "configurable":{
            "session_id":"unused"
        }
    }
).content)

chat_history.messages

作为一个人工智能助手，我实际上并不知道任何关于你的个人信息，除非你之前在与我的对话中主动提供了这些信息。以下是一些可能的情况：

1. **用户名或昵称**：如果你在对话中告诉过我你的名字或昵称，我会知道这一点。
2. **兴趣和爱好**：如果你分享过你的兴趣、爱好或喜欢的活动，我会记得这些信息。
3. **问题和需求**：我会记得你在对话中提出的问题和需求，以便更好地帮助你。

然而，如果你没有提供过任何个人信息，那么我对你一无所知。我的设计原则是尊重用户隐私，不会存储或滥用个人信息。

如果你希望我记住某些信息以便后续对话，你可以明确告诉我，但请确保这些信息是你愿意分享的。

如果你有任何具体问题或需要帮助，请随时告诉我！


[HumanMessage(content='请说出你所知道的关于我的信息'),
 AIMessage(content='根据你刚才的介绍，我知道以下关于你的信息：\n\n1. 你的名字是Chaos。\n2. 你今年22岁。\n3. 你正在学习llm-agent工程。\n\n除此之外，我没有其他关于你的信息。如果你愿意分享更多，我很乐意了解更多关于你的情况！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 59, 'prompt_tokens': 44, 'total_tokens': 103, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'glm-4-plus', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-ab18a99a-d9c3-45e3-af63-3c6dab167b61-0', usage_metadata={'input_tokens': 44, 'output_tokens': 59, 'total_tokens': 103}),
 HumanMessage(content='请说出你所知道的关于我的信息'),
 AIMessage(content='作为一个人工智能助手，我实际上并不知道任何关于你的个人信息，除非你之前在与我的对话中主动提供了这些信息。以下是一些可能的情况：\n\n1. **用户名或昵称**：如果你在对话中告诉过我你的名字或昵称，我会知道这一点。\n2. **兴趣和爱好**：如果你分享过你的兴趣、爱好或喜欢的活动，我会记得这些信息。\n3. **问题和需求**：我会记得你在对话中提出的问题和需求，以便更好地帮助你。\n\n然而，如果你没有提供过任何个人信息，那么我对你一无所知。我的设计原则是尊重用户隐私，不会存储或滥用个人信息。\n\n如果你希望我记住某些信息以便后续对话，你可以明确告诉我，但请确保这些信息是你愿意分享的。\n\n如果你有任何具体问题或需要帮助，请

总结记忆需要调用模型来进行总结，这里会有一个小的chain

In [15]:
def summarize_messages(chain_input):
    stored_messages=chat_history.messages
    # 信息数量大于6才进行总结
    if len(stored_messages)>=6:
        summarization_prompt=ChatPromptTemplate.from_messages(
            [
                MessagesPlaceholder(variable_name="chat_history"),
                ("user","将上述聊天信息提炼成一段摘要信息，尽可能多的包含具体细节")
            ]
        )
        summarization_chain=summarization_prompt|model
        summary_message=summarization_chain.invoke({"chat_history":stored_messages})
        chat_history.clear()
        chat_history.add_message(summary_message)
    return chain_input

chain_with_summarization=summarize_messages|chain_with_memory

chat_history.clear()
chat_history.add_user_message(" 我是chaos")
chat_history.add_ai_message("你好，我是ai，我有什么可以帮您的吗")
chat_history.add_user_message("你觉得我适合做什么")
chat_history.add_ai_message("你适合做ai工程师")
chat_history.add_user_message("为什么")
chat_history.add_ai_message("因为你正在学习ai")

print(chain_with_summarization.invoke(
    {
        'new_message':'你觉得我适合做什么'
    },
    config={
        "configurable":{
            "session_id":"unused"
        }
    }
))
chat_history.messages

content='要确定你适合做什么工作，需要考虑多个因素，包括你的兴趣、技能、经验、个性以及职业目标等。以下是一些常见的问题，可以帮助你自我评估：\n\n1. **兴趣和热情**：\n   - 你对哪些领域特别感兴趣？\n   - 你喜欢什么样的工作环境和任务？\n\n2. **技能和能力**：\n   - 你擅长哪些技能（如编程、沟通、分析等）？\n   - 你有哪些专业知识和经验？\n\n3. **个性特点**：\n   - 你是喜欢与人打交道，还是更倾向于独立工作？\n   - 你是否喜欢面对挑战和解决问题？\n\n4. **职业目标**：\n   - 你对未来职业发展有什么样的期望？\n   - 你希望在工作中获得什么样的成就感和满足感？\n\n如果你能提供更多关于这些方面的信息，我可以给出更具体的建议。例如，如果你对技术感兴趣并且正在学习编程，那么软件开发或数据科学可能是一个不错的选择。如果你喜欢与人沟通并且有较强的组织能力，那么项目管理或市场营销可能更适合你。\n\n当然，职业规划是一个持续的过程，建议你多尝试不同的领域，找到最适合自己的方向。你也可以考虑进行职业测评或咨询职业规划师，以获得更专业的建议。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 245, 'prompt_tokens': 46, 'total_tokens': 291, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'glm-4-plus', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-498168a7-d4f8-48a3-9392-5996f71e89cc-0' usage_metadata={'input_tokens': 46, 'output_tokens': 245, 'total_tokens': 291}


[AIMessage(content='用户“chaos”询问自己适合做什么工作，AI回应认为其适合成为AI工程师，原因是用户正在学习AI领域相关知识。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 60, 'total_tokens': 90, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'glm-4-plus', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-60d32f25-2f08-41ae-a7a6-e97c6e572350-0', usage_metadata={'input_tokens': 60, 'output_tokens': 30, 'total_tokens': 90}),
 HumanMessage(content='你觉得我适合做什么'),
 AIMessage(content='要确定你适合做什么工作，需要考虑多个因素，包括你的兴趣、技能、经验、个性以及职业目标等。以下是一些常见的问题，可以帮助你自我评估：\n\n1. **兴趣和热情**：\n   - 你对哪些领域特别感兴趣？\n   - 你喜欢什么样的工作环境和任务？\n\n2. **技能和能力**：\n   - 你擅长哪些技能（如编程、沟通、分析等）？\n   - 你有哪些专业知识和经验？\n\n3. **个性特点**：\n   - 你是喜欢与人打交道，还是更倾向于独立工作？\n   - 你是否喜欢面对挑战和解决问题？\n\n4. **职业目标**：\n   - 你对未来职业发展有什么样的期望？\n   - 你希望在工作中获得什么样的成就感和满足感？\n\n如果你能提供更多关于这些方面的信息，我可以给出更具体的建议。例如，如果你对技术感兴趣并且正在学习编程，那么软件开发或数据科学可能是一个不错的选择。如果你喜欢与人沟通并且有较强的组织能力，那么项目管理或市场营销可能更适合